# Polynomial Features

**Topic:** Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what polynomial features add to a model and how they are constructed
- **Explain** the tradeoff between polynomial degree and overfitting
- **Interpret** a training vs. validation curve to identify the degree where a model generalizes best

> **Tip:** Watch the validation curve peak around degree 2 and then decline as degree increases further. That peak is the sweet spot where complexity helps without memorizing.

---
## How we got here

In **[04_interaction_features.ipynb](04_interaction_features.ipynb)** you created features that capture joint effects between two variables. In **[ml_concepts/05_overfitting_and_underfitting.ipynb](../ml_concepts/05_overfitting_and_underfitting.ipynb)** you learned what happens when a model becomes too complex for the data.

Polynomial features sit at that exact intersection: they give linear models the ability to learn curves, but every extra degree is another step toward overfitting.

---
## Why this matters for data science

A linear model can only fit straight lines. Many real-world relationships are curved: in this dataset, price doesn't rise or fall steadily as you move east to west across the city — it peaks around Manhattan's narrow geographic band and drops off toward the outer boroughs on either side, a shape a straight line can't capture.

Polynomial features let a linear model capture those curves by adding x², x³, and cross-product terms — without switching to a non-linear algorithm.

---
## Try it yourself

The **Polynomial Fit Explorer** below fits increasing polynomial degrees to a single feature and tracks training vs. validation performance as the degree changes.

### Polynomial Fit Explorer

In [ ]:
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

np.random.seed(42)
_airbnb_w1 = pd.read_csv('../../data/nyc_airbnb.csv')
_airbnb_w1 = _airbnb_w1[(_airbnb_w1['price'] > 0) & (_airbnb_w1['minimum_nights'] <= 30)].copy()
_y_all_w1 = np.log1p(_airbnb_w1['price']).values
_X_all_w1 = _airbnb_w1[['longitude']].values

# Two deliberate choices here, both needed to make the overfitting story visible:
# (1) longitude instead of minimum_nights - minimum_nights only has 30 unique
#     values and barely predicts price (R^2 ~ 2% at any degree); longitude has
#     real, curved signal (NYC's price gradient genuinely bends with geography).
# (2) a 500-row subsample with plain LinearRegression instead of the full ~48k
#     rows with Ridge - on the full dataset, or with Ridge's default alpha,
#     there's too much data/regularization for a single-feature polynomial to
#     meaningfully overfit, so validation R^2 never peaks and declines.
_idx_w1 = np.random.RandomState(42).choice(len(_X_all_w1), 500, replace=False)
_X_w1, _y_w1 = _X_all_w1[_idx_w1], _y_all_w1[_idx_w1]
_Xtr_w1, _Xva_w1, _ytr_w1, _yva_w1 = train_test_split(_X_w1, _y_w1, test_size=0.3, random_state=42)

_degrees_w1 = list(range(1, 9))
_train_r2_w1, _val_r2_w1 = [], []
for d in _degrees_w1:
    pipe = Pipeline([('poly', PolynomialFeatures(d, include_bias=False)),
                      ('scaler', StandardScaler()), ('model', LinearRegression())])
    pipe.fit(_Xtr_w1, _ytr_w1)
    _train_r2_w1.append(pipe.score(_Xtr_w1, _ytr_w1))
    _val_r2_w1.append(pipe.score(_Xva_w1, _yva_w1))

out_poly_fit = Output()

degree_sl_fe = IntSlider(value=2, min=1, max=8, step=1,
    description="Degree:", style={"description_width": "70px"},
    layout=widgets.Layout(width="380px"))

def render_poly_fit(change=None):
    d = degree_sl_fe.value
    pipe = Pipeline([('poly', PolynomialFeatures(d, include_bias=False)),
                      ('scaler', StandardScaler()), ('model', LinearRegression())])
    pipe.fit(_Xtr_w1, _ytr_w1)

    x_line = np.linspace(_X_w1.min(), _X_w1.max(), 200).reshape(-1, 1)
    y_line = pipe.predict(x_line)

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"Degree {d} fit — longitude vs log(price)", "Training vs Validation R² by degree"))

    fig.add_trace(go.Scatter(x=_X_w1[:, 0], y=_y_w1, mode="markers",
        marker=dict(color=PALETTE["muted"], size=4, opacity=0.3), name="Data"), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_line[:, 0], y=y_line, mode="lines",
        line=dict(color=PALETTE["primary"], width=3), name=f"Degree {d} fit"), row=1, col=1)

    fig.add_trace(go.Scatter(x=_degrees_w1, y=_train_r2_w1, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2), name="Training R²"), row=1, col=2)
    fig.add_trace(go.Scatter(x=_degrees_w1, y=_val_r2_w1, mode="lines+markers",
        line=dict(color=PALETTE["secondary"], width=2), name="Validation R²"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[d], y=[_val_r2_w1[d - 1]], mode="markers",
        marker=dict(color=PALETTE["accent"], size=14, symbol="star"),
        name="Selected degree"), row=1, col=2)

    fig.update_xaxes(title_text="longitude", row=1, col=1)
    fig.update_yaxes(title_text="log(price)", row=1, col=1)
    fig.update_xaxes(title_text="Polynomial degree", tickmode="linear", dtick=1, row=1, col=2)
    fig.update_yaxes(title_text="R²", row=1, col=2)
    fig.update_layout(
        height=420, paper_bgcolor=PALETTE["background"], plot_bgcolor=PALETTE["surface"],
        font=dict(family=FONT["family"]),
    )
    with out_poly_fit:
        clear_output(wait=True)
        fig.show()

degree_sl_fe.observe(render_poly_fit, names="value")
display(VBox([degree_sl_fe, out_poly_fit]))
render_poly_fit()


---
## What's happening?

`sklearn.preprocessing.PolynomialFeatures` takes a set of input features and adds all polynomial combinations up to the specified degree. For a single feature x with degree 2, it creates: [1, x, x²]. For two features [x, z] with degree 2: [1, x, z, x², xz, z²].

The number of output features grows fast:

```python
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
# 10 input features + degree 2 = 65 output features
# 10 input features + degree 3 = 285 output features
```

This is why polynomial features are typically applied to a small, curated set of features rather than the entire dataset.

In the widget above, degree 2 captures most of the real curvature in how price varies with longitude, and validation R² peaks there. Past degree 2, training R² keeps climbing — the model is happy to keep adding wiggle — but validation R² drops and then flattens out: the extra flexibility isn't tracking anything real in the relationship, it's just fitting idiosyncrasies of the 500-row training sample.

| Degree | Features created (from 1 input) | Flexibility | Risk | When to use |
|--------|--------------------------------|-------------|------|-------------|
| 1 | x | Low | Underfitting | Baseline |
| 2 | x, x² | Medium | Low | Most real-world curves |
| 3 | x, x², x³ | High | Moderate | Known cubic relationships |
| 4+ | x … xⁿ | Very high | Overfitting | Rarely justified |

---
## Real-world example: longitude vs Price by Polynomial Degree

The widget above already fit a linear model on a single polynomial-expanded feature (`longitude`) across degrees 1 through 8, on a 500-row subsample chosen so real overfitting has room to show up. The cell below reuses those exact same results to call out the best degree explicitly, instead of re-fitting everything from scratch a second time.

In [ ]:
best_degree = _degrees_w1[int(np.argmax(_val_r2_w1))]

print(f"Best degree by validation R²: {best_degree}  (val R² = {_val_r2_w1[best_degree - 1]:.4f})")
print(f"Training R² at that degree:   {_train_r2_w1[best_degree - 1]:.4f}")
print()
print("Degree 1 vs. the selected degree:")
print(f"  degree 1              ->  train R²={_train_r2_w1[0]:.4f}  val R²={_val_r2_w1[0]:.4f}")
print(f"  degree {best_degree} (selected)  ->  train R²={_train_r2_w1[best_degree - 1]:.4f}  "
      f"val R²={_val_r2_w1[best_degree - 1]:.4f}")


> **Discussion question:** If training R² keeps rising with degree but validation R² peaks at degree 2 and then falls, what is happening and what should you do?

---
## Key takeaway

> **Polynomial features let linear models learn curves, but every extra degree is a step toward overfitting — choose the degree where validation performance peaks, not where training performance peaks.**

---
*Next up: 06_handling_skewed_features — where right-skewed price distributions get tamed with log and Box-Cox transforms*